In [ ]:
"""
Sanctions and PEP Screening: Part 2, fuzzy name matching.

Purpose: the SQL baseline (exact and surname matching) misses spelling and
transliteration variants by design. This stage scores how SIMILAR each
customer name is to each watchlist name, so near-matches like Okafor vs
Okafour or Muller vs Mueller can be caught and ranked, and coincidental
collisions rejected by a threshold.

Inputs:
    data/sample_customers.csv   (customer names to screen)
    data/watchlist.csv          (sanctions and PEP names, with aliases)

Output:
    candidate matches with a similarity score, above a chosen threshold.

Dependency: rapidfuzz  ->  pip install rapidfuzz
"""

'\nSanctions and PEP Screening: Part 2, fuzzy name matching.\n\nPurpose: the SQL baseline (exact and surname matching) misses spelling and\ntransliteration variants by design. This stage scores how SIMILAR each\ncustomer name is to each watchlist name, so near-matches like Okafor vs\nOkafour or Muller vs Mueller can be caught and ranked, and coincidental\ncollisions rejected by a threshold.\n\nInputs:\n    data/sample_customers.csv   (customer names to screen)\n    data/watchlist.csv          (sanctions and PEP names, with aliases)\n\nOutput:\n    candidate matches with a similarity score, above a chosen threshold.\n\nDependency: rapidfuzz  ->  pip install rapidfuzz\n'

In [ ]:
import pandas as pd

In [ ]:
# STEP 1: load the two datasets.
# pandas.read_csv reads a CSV into a DataFrame (think of a DataFrame as a
# table, like a SQL result set held in memory).

In [ ]:
df_customer = pd.read_csv(r'C:\Users\t_all\Documents\aml-financial-crime-analytics\data\sample_customers.csv')
df_watchlist = pd.read_csv(r'C:\Users\t_all\Documents\aml-financial-crime-analytics\data\watchlist.csv')

In [ ]:
df_customer.head()


,customer_id,account_id,full_name,customer_type,country,occupation,declared_income,onboarding_channel,pep_flag,account_open_date
0,C0001,100001,Sean Murphy,Individual,Germany,Solicitor,101882,Branch,N,2025-03-02
1,C0002,100002,Noah Muller,Individual,Ireland,Solicitor,55394,Branch,N,2021-08-06
2,C0003,100003,Adam Gallagher,Individual,United Arab Emirates,Software Engineer,83779,Online,N,2022-04-11
3,C0004,100004,Tomasz O'Brien,Individual,Ireland,Consultant,96022,Online,N,2024-02-10
4,C0005,100005,Noah Muller,Business,Ireland,Nurse,36098,Online,N,2025-11-06


In [ ]:
df_watchlist.head()

,watchlist_id,listed_name,watchlist_type,country,alias_1,alias_2
0,WL001,Conor Kowalski,PEP,Ireland,Conor Kowalsky,C. Kowalski
1,WL002,Daniel O'Brien,PEP,Ireland,Daniel Obrien,D. O'Brien
2,WL003,Conor O'Brien,PEP,Italy,Conor Obrien,C. O'Brien
3,WL004,Sean Murphy,SANCTIONS,Germany,Seán Murphy,Sean Murphey
4,WL005,Emma Byrne,SANCTIONS,Ireland,Emma Byrn,E. Byrne


In [ ]:
pip install rapidfuzz

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import rapidfuzz.fuzz as fuzz

In [ ]:
# Sanity check on the scoring function itself, not a real comparison:
# these are just the first row of each dataframe, so an unrelated pair.
# A low score here is expected and confirms fuzz.ratio is working.
score = fuzz.ratio(df_customer['full_name'].iloc[0], df_watchlist['listed_name'].iloc[0])
print(score)

In [ ]:
import rapidfuzz.process as process

print(process.extractOne("Ravi Okafor", ["Ravi Okafour", "Omar Hassan", "Kevin Whelan"]))

('Ravi Okafour', 95.65217391304348, 0)


In [ ]:
# STEP 2 (TODO): score one pair of names.
# rapidfuzz.fuzz.ratio("name a", "name b") returns a similarity score from
# 0 (nothing alike) to 100 (identical). Try it on one known variant pair
# first, so you can see what a "good" score looks like before scaling up.
#
# Example to run once and observe (not the deliverable, just to learn it):
#   print(fuzz.ratio("Ravi Okafor", "Ravi Okafour"))
#   print(fuzz.ratio("Kevin Whelan", "Omar Hassan"))
# Notice the gap between a real variant and two unrelated names.

In [ ]:
df_watchlist["listed_name"]

0        Conor Kowalski
1        Daniel O'Brien
2         Conor O'Brien
3           Sean Murphy
4            Emma Byrne
5           Jack Nguyen
6           Noah Müller
7         Tomasz OBrien
8           Aiofe Patel
9         Chen O Connor
10         Ravi Okafour
11         Gracie Rossi
12    Daniel Mc Carthey
13          Sofie Walsh
14          Omar Hassan
15        Sergey Petrov
16      Fatima Al Zahra
17      Youssef Mansour
18     Viktor Kuznetsov
19         Amina Diallo
Name: listed_name, dtype: str

In [ ]:
print(process.extractOne("Daniel O'Brien", df_watchlist["listed_name"]))

("Daniel O'Brien", 100.0, 1)


In [ ]:
print(process.extractOne("Sean Murphy",  df_watchlist["listed_name"]))

('Sean Murphy', 100.0, 3)


In [ ]:
# extractOne defaults to WRatio, a different (token-aware) scorer than
# fuzz.ratio. Pin scorer=fuzz.ratio here so results stay comparable with
# the plain fuzz.ratio scores used elsewhere in this notebook.
scorer_v2 = process.extractOne("Ravi Okafor", df_watchlist["listed_name"], scorer=fuzz.ratio)
print(scorer_v2)

In [ ]:
# STEP 3 (TODO): for each customer, find the BEST-scoring watchlist match.
# Approach: loop over customers, and for each one compare their full_name to
# every watchlist listed_name, keep the highest score and which entry it was.
# Build a list of results (customer, best match, score, watchlist_type) and
# turn it into a DataFrame.
#
# Guidance, not code: you will need two loops (or an apply), fuzz.ratio for
# the score, and a running "best so far" per customer. Ask if the loop
# structure is unclear and I will walk it through step by step.

In [ ]:
# Compares one customer name against every watchlist name and keeps the
# single highest-scoring match (the "best so far" pattern).
def find_best_watchlist_match(customer_name, watchlist_names):
    best_score = 0
    best_match = None
    for watchlist_name in watchlist_names:
        score = fuzz.ratio(customer_name, watchlist_name)
        if score > best_score:
            best_score = score
            best_match = watchlist_name
    return best_match, best_score

In [ ]:
match, score = find_best_watchlist_match("Ravi Okafor", df_watchlist["listed_name"])
print(f"Best match: {match}, Score: {score}")

Best match: Ravi Okafour, Score: 95.65217391304348


In [ ]:
results = []
for customer_name in df_customer['full_name']:
    best_match, best_score = find_best_watchlist_match(customer_name, df_watchlist["listed_name"])
    results.append((customer_name, best_match, best_score)) 

In [ ]:
print(results[:5])  # print the first 5 results to check

[('Sean Murphy', 'Sean Murphy', 100.0), ('Noah Muller', 'Noah Müller', 90.9090909090909), ('Adam Gallagher', 'Fatima Al Zahra', 48.275862068965516), ("Tomasz O'Brien", 'Tomasz OBrien', 96.2962962962963), ('Noah Muller', 'Noah Müller', 90.9090909090909)]


In [ ]:
similarity_score = pd.DataFrame(results, columns=['customer_name', 'best_watchlist_match', 'similarity_score'])

In [ ]:
similarity_score

,customer_name,best_watchlist_match,similarity_score
0,Sean Murphy,Sean Murphy,100.000000
1,Noah Muller,Noah Müller,90.909091
2,Adam Gallagher,Fatima Al Zahra,48.275862
3,Tomasz O'Brien,Tomasz OBrien,96.296296
4,Noah Muller,Noah Müller,90.909091
...,...,...,...
299,Ravi Silva,Ravi Okafour,54.545455
300,Kevin Whelan,Sean Murphy,34.782609
301,Nova Consulting Ltd,Noah Müller,33.333333
302,Apex Trading Ltd,Aiofe Patel,37.037037


In [ ]:
# STEP 4 (TODO): apply a threshold.
# Filter the results to matches at or above a similarity score you choose.
# The threshold is the real decision here, not the code: too high misses
# transliterations (false negatives, the dangerous error); too low floods
# analysts with coincidental matches (false positives). Pick a number,
# justify it, and be ready to defend the trade-off.

In [ ]:
similarity_score.sort_values(by='similarity_score', ascending=False).head(10)
similarity_score.describe()

,similarity_score
count,304.000000
mean,65.189071
std,12.601540
min,31.250000
25%,59.259259
50%,66.666667
75%,69.565217
max,100.000000


In [ ]:
sorted_results_85 = similarity_score[similarity_score['similarity_score'] >= 85]  # Example threshold of 85

In [ ]:
sorted_results_85

,customer_name,best_watchlist_match,similarity_score
0,Sean Murphy,Sean Murphy,100.000000
1,Noah Muller,Noah Müller,90.909091
3,Tomasz O'Brien,Tomasz OBrien,96.296296
4,Noah Muller,Noah Müller,90.909091
7,Ravi Okafor,Ravi Okafour,95.652174
9,Sophie Walsh,Sofie Walsh,86.956522
11,Emma Byrne,Emma Byrne,100.000000
15,Jack Nguyen,Jack Nguyen,100.000000
17,Aoife Patel,Aiofe Patel,90.909091
19,Grace Rossi,Gracie Rossi,95.652174


In [ ]:
sorted_results_90 = similarity_score[similarity_score['similarity_score'] >= 90]  # Example threshold of 90

In [ ]:
sorted_results_90

,customer_name,best_watchlist_match,similarity_score
0,Sean Murphy,Sean Murphy,100.000000
1,Noah Muller,Noah Müller,90.909091
3,Tomasz O'Brien,Tomasz OBrien,96.296296
4,Noah Muller,Noah Müller,90.909091
7,Ravi Okafor,Ravi Okafour,95.652174
11,Emma Byrne,Emma Byrne,100.000000
15,Jack Nguyen,Jack Nguyen,100.000000
17,Aoife Patel,Aiofe Patel,90.909091
19,Grace Rossi,Gracie Rossi,95.652174
21,Daniel McCarthy,Daniel Mc Carthey,93.750000


# Threshold selected: 90. I selected a similarity threshold of 90 to reduce false-positive screening alerts and analyst review volume. At this level, strong spelling and formatting variants such as Ravi Okafor / Ravi Okafour and Noah Muller / Noah Müller are retained, while weaker similarities are excluded. The trade-off is an increased false-negative risk, including plausible variants below 90, which would require further testing before production use.

In [ ]:
# STEP 5 (TODO): consider aliases.
# The watchlist has alias_1 and alias_2 because listed parties operate under
# alternate names. Decide whether to score against aliases too, and note the
# effect on your matches. This is the gap the SQL baseline could not cover.

In [ ]:
# Extends find_best_watchlist_match to also score against alias_1/alias_2,
# since a listed party's aliases are where the SQL exact-match baseline has
# no coverage. Scores every (listed_name/alias_1/alias_2) value per row and
# keeps the best overall, tracking which field it came from (match_source)
# so an analyst can see whether the hit was on the primary name or an alias.
def find_best_watchlist_match_with_aliases(customer_name, watchlist_df):

    best_score = 0
    best_match = None
    best_watchlist_id = None
    best_watchlist_type = None
    best_match_source = None
    
    for _, row in watchlist_df.iterrows():

        name_options = [
            ("listed_name", row["listed_name"]),
            ("alias_1", row["alias_1"]),
            ("alias_2", row["alias_2"])
        ]

        for source, name in name_options:

            if pd.notna(name):  # skip rows with no alias_1/alias_2 value

                score = fuzz.ratio(customer_name, name)

                if score > best_score:
                    best_score = score
                    best_match = name
                    best_watchlist_id = row["watchlist_id"]
                    best_watchlist_type = row["watchlist_type"]
                    best_match_source = source

    return (
        best_match,
        best_score,
        best_watchlist_id,
        best_watchlist_type,
        best_match_source
    )

In [ ]:
result = find_best_watchlist_match_with_aliases(
    "Ravi Okafor",
    df_watchlist
)

print(result)

('Ravi Okafor', 100.0, 'WL011', 'SANCTIONS', 'alias_1')


In [ ]:
results_alias = []

for customer_name in df_customer["full_name"]:

    matched_name, score, watchlist_id, watchlist_type, match_source = \
        find_best_watchlist_match_with_aliases(
            customer_name,
            df_watchlist
        )

    results_alias.append((
        customer_name,
        matched_name,
        score,
        watchlist_id,
        watchlist_type,
        match_source
    ))

In [ ]:
print(results_alias[:5])

[('Sean Murphy', 'Sean Murphy', 100.0, 'WL004', 'SANCTIONS', 'listed_name'), ('Noah Muller', 'Noah Muller', 100.0, 'WL007', 'SANCTIONS', 'alias_1'), ('Adam Gallagher', 'Fatima Al Zahra', 48.275862068965516, 'WL017', 'PEP', 'listed_name'), ("Tomasz O'Brien", "Tomasz O'Brien", 100.0, 'WL008', 'PEP', 'alias_1'), ('Noah Muller', 'Noah Muller', 100.0, 'WL007', 'SANCTIONS', 'alias_1')]


In [ ]:
alias_results = pd.DataFrame(results_alias, columns=['customer_name', 'best_watchlist_match', 'similarity_score', 'watchlist_id', 'watchlist_type', 'match_source'])

In [ ]:
alias_results

,customer_name,best_watchlist_match,similarity_score,watchlist_id,watchlist_type,match_source
0,Sean Murphy,Sean Murphy,100.000000,WL004,SANCTIONS,listed_name
1,Noah Muller,Noah Muller,100.000000,WL007,SANCTIONS,alias_1
2,Adam Gallagher,Fatima Al Zahra,48.275862,WL017,PEP,listed_name
3,Tomasz O'Brien,Tomasz O'Brien,100.000000,WL008,PEP,alias_1
4,Noah Muller,Noah Muller,100.000000,WL007,SANCTIONS,alias_1
...,...,...,...,...,...,...
299,Ravi Silva,Ravi Okafor,57.142857,WL011,SANCTIONS,alias_1
300,Kevin Whelan,Sean Murphey,41.666667,WL004,SANCTIONS,alias_2
301,Nova Consulting Ltd,Noah Muller,40.000000,WL007,SANCTIONS,alias_1
302,Apex Trading Ltd,Aiofe Patel,37.037037,WL009,SANCTIONS,listed_name


In [ ]:
alias_results_90 = alias_results[alias_results['similarity_score'] >= 90]  # Example threshold of 90

In [ ]:
alias_results_90

,customer_name,best_watchlist_match,similarity_score,watchlist_id,watchlist_type,match_source
0,Sean Murphy,Sean Murphy,100.0,WL004,SANCTIONS,listed_name
1,Noah Muller,Noah Muller,100.0,WL007,SANCTIONS,alias_1
3,Tomasz O'Brien,Tomasz O'Brien,100.0,WL008,PEP,alias_1
4,Noah Muller,Noah Muller,100.0,WL007,SANCTIONS,alias_1
7,Ravi Okafor,Ravi Okafor,100.0,WL011,SANCTIONS,alias_1
9,Sophie Walsh,Sophie Walsh,100.0,WL014,PEP,alias_1
11,Emma Byrne,Emma Byrne,100.0,WL005,SANCTIONS,listed_name
15,Jack Nguyen,Jack Nguyen,100.0,WL006,SANCTIONS,listed_name
17,Aoife Patel,Aoife Patel,100.0,WL009,SANCTIONS,alias_1
19,Grace Rossi,Grace Rossi,100.0,WL012,PEP,alias_1


In [ ]:
# Count of strong (>=90) matches with vs without alias scoring, to quantify
# how many additional true matches alias-aware screening recovers.
alias_01 = alias_results_90.shape[0]
listed_01 = sorted_results_90.shape[0]

In [ ]:
print(listed_01)
print(alias_01)

20
22


# Alias-aware screening increased strong candidate matches from 20 to 22 at a threshold of 90, demonstrating that alternative names can recover matches missed by primary-name-only screening.

In [ ]:
alias_results_90[alias_results_90['customer_name'] == 'Ravi Okafor']

,customer_name,best_watchlist_match,similarity_score,watchlist_id,watchlist_type,match_source
7,Ravi Okafor,Ravi Okafor,100.0,WL011,SANCTIONS,alias_1


In [ ]:
# SELF-CHECK (after you have output and a threshold):
# Did fuzzy matching catch the planted variants the SQL baseline missed
# (Okafor/Okafour, Aoife/Aiofe Patel, Muller/Mueller)? Did your threshold
# reject unrelated same-surname collisions? Record the threshold chosen and
# the trade-off accepted.